## Summary

This notebook validates Azure Cosmos DB access using a service principal.

- Configures the Cosmos DB Spark connector with service principal authentication
- Reads data from the `customers` container
- Tests change feed access
- Verifies that write operations are blocked for read-only access


In [0]:
TENANT_ID       = "f4684704-cde2-4f4d-8eb0-be045ed13a2c"
SUBSCRIPTION_ID = "d93f4f31-c264-4bb9-aa6a-efa162124a9e"

cosmos_cfg_sp = {
    "spark.cosmos.accountEndpoint": "https://cosmos-dbx-poc-sm.documents.azure.com:443/",
    "spark.cosmos.auth.type": "ServicePrincipal",
    "spark.cosmos.account.tenantId": TENANT_ID,
    "spark.cosmos.account.subscriptionId": SUBSCRIPTION_ID,
    "spark.cosmos.account.resourceGroupName": "rg-cosmos-dbx-poc",
    "spark.cosmos.auth.aad.clientId": dbutils.secrets.get("kv-poc", "sp-cosmos-client-id"),
    "spark.cosmos.auth.aad.clientSecret": dbutils.secrets.get("kv-poc", "sp-cosmos-client-secret"),
    "spark.cosmos.database": "pocdb",
    "spark.cosmos.container": "customers",
    "spark.cosmos.read.inferSchema.enabled": "true",
}

In [0]:
df = spark.read.format("cosmos.oltp").options(**cosmos_cfg_sp).load()
print("Rows read as SP:", df.count())
display(df.select("id", "customerId", "name", "tier", "tenant"))

In [0]:
cf = (spark.read.format("cosmos.oltp.changeFeed")
      .options(**cosmos_cfg_sp)
      .option("spark.cosmos.changeFeed.mode", "LatestVersion")
      .option("spark.cosmos.changeFeed.startFrom", "Beginning")
      .option("spark.cosmos.read.inferSchema.includeTimestamp", "true")
      .load())
print("Change feed rows as SP:", cf.count())
display(cf.select("id", "customerId", "_ts"))

In [0]:
from pyspark.sql import Row

try:
    (spark.createDataFrame([Row(id="99", customerId="C099", name="Should Fail", tier="None", tenant="A")])
        .write.format("cosmos.oltp")
        .options(**cosmos_cfg_sp)
        .option("spark.cosmos.write.strategy", "ItemOverwrite")
        .mode("append")
        .save())
    print("UNEXPECTED: write succeeded - read-only NOT enforced")
except Exception as e:
    msg = str(e)
    blocked = ("403" in msg) or ("Forbidden" in msg)
    print("RESULT:", "Write blocked as expected (read-only enforced)" if blocked else "Write failed for a different reason - review below")
    print(msg[:1500])